In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HOQ_C.xpt"
csv_file = DATA_DIR / "HOQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 HOQ_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nVariables:")
print(df.columns.tolist())


# ============================================================
# 3. EXPECTED RELEASED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HOD011",
    "HOD030",
    "HOD040",
    "HOD050",
    "HOD060",
    "HOQ065",
    "HOQ070",
    "HOQ080",
    "HOD140",
    "HOD150",
    "HOD160",
    "HOD170",
    "HOD180",
    "HOD190",
    "HOD210",
    "HOD220"
]


if df.columns.tolist() == EXPECTED_COLUMNS:
    print("\nPASS: Column names/order match HOQ_C.")
else:
    raise ValueError(
        "HOQ_C column structure differs from expected."
    )


# ============================================================
# 4. CHECK KNOWN PANDAS/XPT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


total_tiny = int(
    tiny_counts.sum()
)

total_zero = int(
    zero_counts.sum()
)


print("\n--- ORIGINAL ZERO CHECK ---")

print(
    "True numeric zeros:",
    total_zero
)

print(
    "Tiny artifact values:",
    total_tiny
)


if total_tiny > 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
    )

    # IMPORTANT:
    # No tiny-value restoration is expected for this HOQ_C XPT.
    raise ValueError(
        "STOP: Unexpected tiny-value artifact found in HOQ_C. "
        "Do NOT automatically replace it with zero."
    )


print(
    "PASS: No tiny zero-decoding artifact exists."
)


# ============================================================
# 5. VERIFY THERE ARE NO GENUINE DECIMALS
# ============================================================

#
# Fidelity rule:
#
# Never round a real decimal.
#
# Before converting any field to Int64, verify that
# every source value is a whole number.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values)
    )

    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nGenuine decimal values found in {col}:"
        )

        print(
            values[
                fractional_mask
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values were found. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: All HOQ_C source values are "
    "whole-number codes/quantities."
)


# ============================================================
# 6. CLEAN SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn)
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 7. FORMAT HOQ/HOD FIELDS WITHOUT .0
# ============================================================

#
# All these fields were already checked above.
# There are no genuine decimals.
#

CODE_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


print(
    "\nPASS: HOQ/HOD fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 8. CDC STRUCTURE CHECK
# ============================================================

print("\n--- CDC STRUCTURE CHECK ---")


if len(df) == 10122:
    print("PASS: Row count = 10,122")
else:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) == 17:
    print("PASS: Column count = 17")
else:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


# ============================================================
# 9. CDC EXACT FREQUENCY CHECKS
# ============================================================

EXPECTED = {

    "HOD011": {
        "counts": {
            1:711,
            2:5730,
            3:919,
            4:2466,
            5:63,
            6:95,
            77:2,
            99:3
        },
        "missing":133
    },

    "HOD030": {
        "counts": {
            1:367,
            3:443,
            4:454,
            5:487,
            6:284,
            7:358,
            77:8,
            99:59
        },
        "missing":7662
    },

    "HOD040": {
        "counts": {
            1:1305,
            2:1302,
            3:1903,
            4:1077,
            5:459,
            6:1142,
            77:18,
            99:2783
        },
        "missing":133
    },

    "HOD060": {
        "counts": {
            1:1801,
            2:1878,
            3:1985,
            4:1592,
            5:2717,
            77:12,
            99:4
        },
        "missing":133
    },

    "HOQ065": {
        "counts": {
            1:5760,
            2:3960,
            3:265,
            7:3,
            9:1
        },
        "missing":133
    },

    "HOQ070": {
        "counts": {
            1:8914,
            2:950,
            3:22,
            7:2,
            9:101
        },
        "missing":133
    },

    "HOQ080": {
        "counts": {
            1:2226,
            2:7726,
            7:2,
            9:35
        },
        "missing":133
    },

    "HOD140": {
        "counts": {
            1:527,
            2:694,
            9:23
        },
        "missing":8878
    },

    "HOD150": {
        "counts": {
            1:122,
            2:335,
            7:2,
            9:68
        },
        "missing":9595
    },

    "HOD160": {
        "counts": {
            1:360,
            2:882,
            9:2
        },
        "missing":8878
    },

    "HOD170": {
        "counts": {
            1:62,
            2:298
        },
        "missing":9762
    },

    "HOD180": {
        "counts": {
            1:41,
            2:21
        },
        "missing":10060
    },

    "HOD190": {
        "counts": {
            1:216,
            2:1007,
            9:21
        },
        "missing":8878
    },

    "HOD210": {
        "counts": {
            1:53,
            2:162,
            9:1
        },
        "missing":9906
    },

    "HOD220": {
        "counts": {
            1:198,
            2:1019,
            9:27
        },
        "missing":8878
    }
}


print("\n--- CDC FREQUENCY CHECK ---")


frequency_failures = []


for col, spec in EXPECTED.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if (
        actual_counts == spec["counts"]
        and
        actual_missing == spec["missing"]
    ):

        print(
            f"PASS: {col}"
        )

    else:

        frequency_failures.append(
            col
        )

        print(
            f"\nFAIL: {col}"
        )

        print(
            "Expected:",
            spec["counts"]
        )

        print(
            "Actual:",
            actual_counts
        )

        print(
            "Expected missing:",
            spec["missing"]
        )

        print(
            "Actual missing:",
            actual_missing
        )


if frequency_failures:

    raise ValueError(
        "CDC frequency mismatch in:\n"
        + str(frequency_failures)
    )


print(
    "\nPASS: All categorical HOQ_C "
    "variables match CDC."
)


# ============================================================
# 10. SPECIAL CHECK: HOD050
# ============================================================

#
# CDC:
#
# 1-12    = 9,944
# 13      = 27       (13 or more)
# 777     = 10
# 999     = 8
# Missing = 133
#

s = df["HOD050"]


regular = int(
    s.between(
        1,
        12
    ).sum()
)

topcoded = int(
    s.eq(13).sum()
)

refused = int(
    s.eq(777).sum()
)

dont_know = int(
    s.eq(999).sum()
)

missing = int(
    s.isna().sum()
)


print("\n--- HOD050 ---")

print("1-12:", regular)
print("13:", topcoded)
print("777:", refused)
print("999:", dont_know)
print("Missing:", missing)


if (
    regular == 9944
    and
    topcoded == 27
    and
    refused == 10
    and
    dont_know == 8
    and
    missing == 133
):

    print(
        "PASS: HOD050 matches CDC."
    )

else:

    raise ValueError(
        "HOD050 does not match CDC."
    )


# ============================================================
# 11. VERIFY ORIGINAL XPT VALUES WERE NOT CHANGED
# ============================================================

#
# No zero correction occurs in HOQ_C.
#
# Therefore the numeric values must match the
# source XPT exactly.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


for col in source_df.columns:

    source_values = (
        source_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        source_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} preserved."
        )

    else:

        raise ValueError(
            f"FAIL: {col} changed."
        )


# ============================================================
# 12. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 13. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 14. EXACT XPT -> CSV VALUE CHECK
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


value_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )

    csv_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        csv_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} matches XPT."
        )

    else:

        value_failures.append(
            col
        )

        print(
            f"FAIL: {col} differs."
        )


if value_failures:

    raise ValueError(
        "XPT -> CSV differences in:\n"
        + str(value_failures)
    )


print(
    "\nPASS: Every HOQ_C value matches "
    "the source XPT."
)


# ============================================================
# 15. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            csv_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches == 0:

    print(
        "PASS: Missing values preserved exactly."
    )

else:

    raise ValueError(
        "Missing-value positions changed."
    )


# ============================================================
# 16. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    csv_df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny artifact values:",
    remaining_tiny
)


if remaining_tiny == 0:

    print(
        "PASS: No tiny zero artifact."
    )

else:

    raise ValueError(
        "Unexpected tiny value exists."
    )


# ============================================================
# 17. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count == 0:

    print(
        "PASS: No unnecessary .0 formatting."
    )

else:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific-notation occurrences:",
    scientific_count
)


if scientific_count == 0:

    print(
        "PASS: No scientific notation."
    )


print("\n" + "=" * 75)
print("HOQ_C CONVERSION COMPLETE")
print("=" * 75)

NHANES 2003-2004 HOQ_C XPT -> CSV
Rows: 10,122
Columns: 17

Variables:
['SEQN', 'HOD011', 'HOD030', 'HOD040', 'HOD050', 'HOD060', 'HOQ065', 'HOQ070', 'HOQ080', 'HOD140', 'HOD150', 'HOD160', 'HOD170', 'HOD180', 'HOD190', 'HOD210', 'HOD220']

PASS: Column names/order match HOQ_C.

--- ORIGINAL ZERO CHECK ---
True numeric zeros: 0
Tiny artifact values: 0
PASS: No tiny zero-decoding artifact exists.

--- FRACTIONAL VALUE CHECK ---
PASS: All HOQ_C source values are whole-number codes/quantities.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31126

PASS: HOQ/HOD fields formatted without unnecessary .0.

--- CDC STRUCTURE CHECK ---
PASS: Row count = 10,122
PASS: Column count = 17

--- CDC FREQUENCY CHECK ---
PASS: HOD011
PASS: HOD030
PASS: HOD040
PASS: HOD060
PASS: HOQ065
PASS: HOQ070
PASS: HOQ080
PASS: HOD140
PASS: HOD150
PASS: HOD160
PASS: HOD170
PASS: HOD180
PASS: HOD190
PASS: HOD210
PASS: HOD220

PASS: All categorical HOQ_C variables match CDC.

--- HOD050 ---
1-12: 9944
